# Duka Electronics: Half-Year Sales Exploration (January to June 2026)

**Analyst:** Marion Oduor · PLP Data Analytics 2026 Cohort

The Head of Sales wants answers, not just tables, ahead of Thursday's half-year review. This notebook explores the Duka Electronics order data (five branches, five products, three sales channels) and answers each question in plain language beneath the code.

## Executive Summary

- Duka Electronics earned KES 71.8 million from 1,000 orders between 1 January and 29 June 2026.
- Nairobi is the anchor branch with 36.4% of revenue (KES 26.1 million), almost double second-placed Mombasa at 19.5%, and the branch ranking is the same whether measured by revenue or by number of orders.
- Premium products drive the business: Laptops and Tablets alone generate 84.8% of revenue, and Laptops bring in KES 41.5 million even though Tablets sell more units (691 against 639).
- Weekends are not stronger than weekdays once the number of days is considered: weekend days average KES 383,390 in revenue against KES 405,127 on weekdays, while the average order value is almost identical (KES 71,972 versus KES 71,724).
- April was the best month at KES 13.5 million and June the lowest at KES 10.9 million, a 24.0% gap (though June has only 29 days of data), and the 83 Premium online orders from Nairobi and Mombasa are worth KES 10.2 million (14.2% of revenue), a segment worth protecting and growing.

## Setup and data loading

All imports are in one cell. If the data file is missing (for example in a fresh Colab session), the cell downloads the repository files and runs `generate_data.py`, which recreates `duka_sales.csv` with the fixed seed 2026.

In [1]:
import os
import subprocess
import sys
import urllib.request

import numpy as np
import pandas as pd

REPO_RAW = ("https://raw.githubusercontent.com/Reeescyberspacexx/"
            "plp-da-a3-duka-sales-and-agent-cleanup/main/")


def ensure_file(path):
    """Download a repository file if it is not already available locally."""
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + path, path)


ensure_file("generate_data.py")
ensure_file("excel/duka_branch_summary.xlsx")
if not os.path.exists("duka_sales.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)
os.makedirs("outputs", exist_ok=True)

In [2]:
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

## Task 1: First-Five-Minutes Inspection

**Approach:** Run the same inspection routine on every new dataset before analysing anything: look at the first and last rows, then the shape, data types, missing values, duplicates, summary statistics, date range and the mix of the key categories.

In [3]:
print("Shape (rows, columns):", sales.shape)
display(sales.head())
display(sales.tail())
sales.info()
display(sales.describe())

Shape (rows, columns): (1000, 7)


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[us]
 2   branch      1000 non-null   str           
 3   product     1000 non-null   str           
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   str           
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 54.8 KB


,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546


In [4]:
print("Missing values per column:")
print(sales.isna().sum())
print("\nDuplicate order_id values:", sales["order_id"].duplicated().sum())
print("\nDistinct values per column:")
print(sales.nunique())
print(f"\nPeriod covered: {sales['order_date'].min():%d %b %Y} to {sales['order_date'].max():%d %b %Y}")
print("\nBranch mix (share of orders):")
print(sales["branch"].value_counts(normalize=True).round(3))
print("\nChannel mix (share of orders):")
print(sales["channel"].value_counts(normalize=True).round(3))
print("\nProduct counts:")
print(sales["product"].value_counts())

Missing values per column:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Duplicate order_id values: 0

Distinct values per column:
order_id      1000
order_date     179
branch           5
product          5
quantity         5
channel          3
unit_price       5
dtype: int64

Period covered: 01 Jan 2026 to 29 Jun 2026

Branch mix (share of orders):
branch
Nairobi    0.376
Mombasa    0.186
Nakuru     0.162
Kisumu     0.159
Eldoret    0.117
Name: proportion, dtype: float64

Channel mix (share of orders):
channel
In-store       0.579
Online         0.320
Phone order    0.101
Name: proportion, dtype: float64

Product counts:
product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184
Name: count, dtype: int64


**What the inspection shows**

- **Shape:** 1,000 rows and 7 columns, one row per order.
- **Types:** `order_date` was parsed as a real datetime because of `parse_dates`; `order_id`, `quantity` and `unit_price` are integers; `branch`, `product` and `channel` are text categories.
- **Missing values and duplicates:** none. There are 0 missing values in every column and 0 duplicated `order_id` values, so no cleaning was needed.
- **Date range:** 01 January 2026 to 29 June 2026 (180 calendar days). The data stops on 29 June, so June is one day short of a full month, which matters when comparing months in Task 6.
- **Key categories:** five branches (Nairobi has the largest share of orders at 37.6%), five products (Laptop is the most frequently ordered at 220 orders and Charger the least at 184) and three channels (In-store 57.9%, Online 32.0%, Phone order 10.1% of orders).
- **Ranges:** quantity runs from 1 to 5 per order and unit prices from KES 1,200 (Charger) to KES 65,000 (Laptop), so one laptop order is worth far more than many charger orders. This is why order counts and revenue can tell different stories.

## Task 2: Derived Columns

**Approach:** Create `revenue` (quantity × unit price), `month` and `weekday` names with the `.dt` accessor, the boolean `is_weekend`, and `price_band` with `np.where` (Premium for a unit price of KES 28,000 or more, otherwise Value). `value_counts()` checks that each new column looks sensible.

In [5]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5          # Saturday = 5, Sunday = 6
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")

display(sales.head())
print(sales["is_weekend"].value_counts(), "\n")
print(sales["price_band"].value_counts(), "\n")
print(sales["weekday"].value_counts())
print(f"\nTotal revenue: KES {sales['revenue'].sum():,}")

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


is_weekend
False    723
True     277
Name: count, dtype: int64 

price_band
Value      565
Premium    435
Name: count, dtype: int64 

weekday
Friday       166
Sunday       145
Thursday     145
Monday       140
Wednesday    137
Tuesday      135
Saturday     132
Name: count, dtype: int64

Total revenue: KES 71,792,500


**Check:** total revenue across all orders is KES 71,792,500. 435 orders are Premium (Laptops and Tablets) and 565 are Value (Phones, Headphones and Chargers). Phones at KES 15,000 fall below the Premium threshold of KES 28,000.

## Task 3: Branch Ranking by Revenue and by Orders

**Approach:** Group by branch and calculate total revenue, number of orders and average order value, then rank the branches by revenue and by orders and compare the two rankings.

In [6]:
branch_summary = sales.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    orders=("order_id", "count"),
    avg_order_value=("revenue", "mean"),
)
branch_summary["revenue_share_pct"] = (
    branch_summary["total_revenue"] / branch_summary["total_revenue"].sum() * 100).round(1)
branch_summary["revenue_rank"] = branch_summary["total_revenue"].rank(ascending=False).astype(int)
branch_summary["orders_rank"] = branch_summary["orders"].rank(ascending=False).astype(int)
branch_summary = branch_summary.sort_values("total_revenue", ascending=False)
branch_summary["avg_order_value"] = branch_summary["avg_order_value"].round(0)

display(branch_summary)
print("Rankings identical:", (branch_summary["revenue_rank"] == branch_summary["orders_rank"]).all())

,total_revenue,orders,avg_order_value,revenue_share_pct,revenue_rank,orders_rank
branch,,,,,,
Nairobi,26106100,376,69431.0,36.4,1,1
Mombasa,14003700,186,75289.0,19.5,2,2
Nakuru,11970400,162,73891.0,16.7,3,3
Kisumu,10868100,159,68353.0,15.1,4,4
Eldoret,8844200,117,75591.0,12.3,5,5


Rankings identical: True


**Answer:** Yes, the two rankings are identical: Nairobi first, then Mombasa, Nakuru, Kisumu and Eldoret.

- **Nairobi** leads on both measures with 376 orders and KES 26,106,100 (36.4% of revenue), almost double Mombasa (KES 14,003,700, 19.5%).
- **Why they agree:** revenue is orders multiplied by average order value, and the average order values are close together (from KES 68,353 to KES 75,591, a spread of only 11%). With prices that similar, the number of orders decides the ranking.
- **A small difference underneath:** Nakuru and Kisumu have almost the same number of orders (162 against 159), so Nakuru's lead of KES 1,102,300 comes from a higher average order value (KES 73,891 against KES 68,353), not from more orders. The rankings match here, but the gap between these two branches is small (about 17 laptops' worth of revenue).

## Task 4: Top Three Products by Revenue and by Units Sold

**Approach:** Group by product to total revenue and units (the sum of `quantity`), then use `nlargest(3, ...)` on each measure and compare the two lists.

In [7]:
product_summary = sales.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    units_sold=("quantity", "sum"),
    unit_price=("unit_price", "first"),
)

print("Top 3 by revenue:")
display(product_summary.nlargest(3, "total_revenue"))
print("Top 3 by units sold:")
display(product_summary.nlargest(3, "units_sold"))
print("All products, ranked by revenue:")
display(product_summary.sort_values("total_revenue", ascending=False))

Top 3 by revenue:


,total_revenue,units_sold,unit_price
product,,,
Laptop,41535000,639,65000
Tablet,19348000,691,28000
Phone,8880000,592,15000


Top 3 by units sold:


,total_revenue,units_sold,unit_price
product,,,
Tablet,19348000,691,28000
Laptop,41535000,639,65000
Phone,8880000,592,15000


All products, ranked by revenue:


,total_revenue,units_sold,unit_price
product,,,
Laptop,41535000,639,65000
Tablet,19348000,691,28000
Phone,8880000,592,15000
Headphones,1387500,555,2500
Charger,642000,535,1200


**Answer:** The two lists contain the same three products but in a different order.

- **By revenue:** Laptop (KES 41,535,000), Tablet (KES 19,348,000) and Phone (KES 8,880,000).
- **By units sold:** Tablet (691 units), Laptop (639) and Phone (592).
- **Why the first two swap places:** revenue is units multiplied by price. Tablets sold 52 more units than Laptops, but a Laptop costs KES 65,000 against KES 28,000, so Laptops earn 2.1 times the revenue of Tablets.
- **The rest:** Headphones and Charger sell 555 and 535 units, but their low prices (KES 2,500 and KES 1,200) keep them to KES 1,387,500 and KES 642,000 in revenue.

Units show what customers buy most often; revenue shows what pays the bills. For stock and cash planning the revenue list matters more.

## Task 5: Weekend versus Weekday Orders

**Approach:** Group by `is_weekend` to compare orders, total revenue and average order revenue. Because a week has five weekdays and only two weekend days, raw totals favour weekdays, so the number of calendar days is counted and per-day figures are added for a fair comparison.

In [8]:
calendar = pd.Series(pd.date_range(sales["order_date"].min(), sales["order_date"].max()))
days_count = (calendar.dt.dayofweek >= 5).value_counts()          # True = weekend days

weekend_summary = sales.groupby("is_weekend").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_order_revenue=("revenue", "mean"),
)
weekend_summary["calendar_days"] = days_count
weekend_summary["orders_per_day"] = (weekend_summary["orders"] / weekend_summary["calendar_days"]).round(2)
weekend_summary["revenue_per_day"] = (weekend_summary["total_revenue"] / weekend_summary["calendar_days"]).round(0)
weekend_summary["avg_order_revenue"] = weekend_summary["avg_order_revenue"].round(0)
weekend_summary.index = weekend_summary.index.map({False: "Weekday (Mon-Fri)", True: "Weekend (Sat-Sun)"})
display(weekend_summary)

,orders,total_revenue,avg_order_revenue,calendar_days,orders_per_day,revenue_per_day
is_weekend,,,,,,
Weekday (Mon-Fri),723,51856200,71724.0,128,5.65,405127.0
Weekend (Sat-Sun),277,19936300,71972.0,52,5.33,383390.0


**Answer:** Weekdays bring in more in total, but that is only because there are more of them.

- **Totals:** weekdays had 723 orders and KES 51,856,200 in revenue, against 277 orders and KES 19,936,300 on weekends.
- **Average order revenue** is practically the same: KES 71,724 on weekdays and KES 71,972 on weekends, so weekend customers do not spend more per order.
- **Per day** (the fair comparison, with 128 weekdays and 52 weekend days in the period): weekdays average 5.65 orders and KES 405,127 a day, while weekend days average 5.33 orders and KES 383,390 a day. Weekend revenue per day is 5.4% lower than weekdays.
- **Meaning for the business:** this data shows no weekend sales peak, so weekend staffing and promotions should not be assumed to pay for themselves.

## Task 6: Best and Worst Month for Revenue

**Approach:** Group revenue by month and put the months in calendar order. Because the data ends on 29 June, the number of days covered in each month is counted and revenue per day is added to check that the ranking is not just a calendar effect.

In [9]:
month_order = ["January", "February", "March", "April", "May", "June"]
monthly = sales.groupby("month").agg(total_revenue=("revenue", "sum"), orders=("order_id", "count"))
monthly = monthly.loc[month_order]
monthly["days_covered"] = calendar.dt.month_name().value_counts().loc[month_order]
monthly["revenue_per_day"] = (monthly["total_revenue"] / monthly["days_covered"]).round(0)
display(monthly)

best_month, worst_month = monthly["total_revenue"].idxmax(), monthly["total_revenue"].idxmin()
best_rev, worst_rev = monthly["total_revenue"].max(), monthly["total_revenue"].min()
print(f"Best month:  {best_month} (KES {best_rev:,})")
print(f"Worst month: {worst_month} (KES {worst_rev:,})")
print(f"{best_month} is {(best_rev - worst_rev) / worst_rev * 100:.1f}% higher than {worst_month}")
print(f"{worst_month} is {(best_rev - worst_rev) / best_rev * 100:.1f}% lower than {best_month}")

,total_revenue,orders,days_covered,revenue_per_day
month,,,,
January,11786600,173,31,380213.0
February,11772000,159,28,420429.0
March,12396800,186,31,399897.0
April,13518300,167,30,450610.0
May,11415400,153,31,368239.0
June,10903400,162,29,375979.0


Best month:  April (KES 13,518,300)
Worst month: June (KES 10,903,400)
April is 24.0% higher than June
June is 19.3% lower than April


**Answer:** April was the best month with KES 13,518,300 and June was the worst with KES 10,903,400.

- **The gap:** April was **24.0% higher** than June, or put the other way, June was 19.3% lower than April.
- **Fair-comparison check:** June has only 29 days of data (the data stops on 29 June), against 31 days in May. Per day, April still leads clearly with KES 450,610 a day, but the weakest month becomes May (KES 368,239 a day) rather than June (KES 375,979 a day). So June is the lowest month in total mainly because it is a shorter month in this dataset, not because its daily sales were the weakest.
- **Caution:** with only six months of data there is no way to tell whether this is a seasonal pattern or ordinary variation. The leadership team should look at the same months in 2025 before acting on it.

## Task 7: Premium Online Orders from Nairobi and Mombasa

**Approach:** Combine three conditions with `&`, wrapping each in parentheses: price band is Premium, channel is Online and the branch is Nairobi or Mombasa (using `isin`). The result is exported with `index=False`.

In [10]:
premium_online = sales[
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
]

print("Number of orders:", len(premium_online))
print(f"Total revenue: KES {premium_online['revenue'].sum():,}")
print(f"Share of all revenue: {premium_online['revenue'].sum() / sales['revenue'].sum() * 100:.1f}%")
print(premium_online["product"].value_counts().to_string())

premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)
check = pd.read_csv("outputs/premium_online_coast_capital.csv")
print("\nSaved outputs/premium_online_coast_capital.csv with", len(check), "rows and columns:", list(check.columns))

Number of orders: 83
Total revenue: KES 10,218,000
Share of all revenue: 14.2%
product
Tablet    50
Laptop    33

Saved outputs/premium_online_coast_capital.csv with 83 rows and columns: ['order_id', 'order_date', 'branch', 'product', 'quantity', 'channel', 'unit_price', 'revenue', 'month', 'weekday', 'is_weekend', 'price_band']


**Answer:** There are **83 Premium online orders** from Nairobi and Mombasa, with a combined revenue of **KES 10,218,000**, which is 14.2% of all revenue. They have been exported to `outputs/premium_online_coast_capital.csv`.

- **Mix:** 33 of the orders are Laptops and 50 are Tablets, and the average order is worth KES 123,108, well above the overall average of KES 71,792.
- **Why it matters:** online orders account for 31.7% of all revenue, and these Premium orders from Nairobi and Mombasa make up 44.8% of all online revenue. Because each order is worth so much more than average, this segment is worth tracking separately.

## Task 8: Spreadsheet Check (Excel)

**Approach:** The workbook `excel/duka_branch_summary.xlsx` contains the sales data with a revenue formula and an XLOOKUP region column, a SUMIFS branch revenue table, and a PivotTable of revenue by branch and channel. Here the workbook is read back and its totals are compared with the pandas results.

In [11]:
xl_path = "excel/duka_branch_summary.xlsx"

# 1. SUMIFS branch table in the workbook versus pandas
xl_branch = pd.read_excel(xl_path, sheet_name="Branch Summary", nrows=5).set_index("branch")
pandas_branch = sales.groupby("branch")["revenue"].sum()
sumifs_match = (xl_branch["revenue_kes"] == pandas_branch.reindex(xl_branch.index)).all()
orders_match = (xl_branch["orders"] == sales["branch"].value_counts().reindex(xl_branch.index)).all()

# 2. PivotTable (branch x channel) versus pandas
xl_pivot = pd.read_excel(xl_path, sheet_name="PivotTable", header=5).set_index("branch")
grand_total = xl_pivot.loc["Total Result", "Total Result"]
xl_pivot = xl_pivot.drop(index="Total Result", columns="Total Result")
pandas_pivot = sales.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum")
pivot_match = (xl_pivot.sort_index().astype(int)
               == pandas_pivot.sort_index().astype(int)).all().all()

# 3. XLOOKUP region column versus the branch-to-region mapping
xl_sales = pd.read_excel(xl_path, sheet_name="Sales")
region_map = {"Nairobi": "Central", "Mombasa": "Coast", "Kisumu": "Western",
              "Nakuru": "Rift Valley", "Eldoret": "North Rift"}
region_match = (xl_sales["region"] == xl_sales["branch"].map(region_map)).all()

print("SUMIFS branch revenue matches pandas:", sumifs_match)
print("COUNTIFS order counts match pandas:  ", orders_match)
print("PivotTable branch x channel matches: ", pivot_match)
print("XLOOKUP region column is correct:    ", region_match)
print(f"Grand total: Excel KES {grand_total:,} | pandas KES {sales['revenue'].sum():,} "
      f"| equal: {grand_total == sales['revenue'].sum()}")
display(xl_branch)
display(xl_pivot)

SUMIFS branch revenue matches pandas: True
COUNTIFS order counts match pandas:   True
PivotTable branch x channel matches:  True
XLOOKUP region column is correct:     True
Grand total: Excel KES 71,792,500 | pandas KES 71,792,500 | equal: True


,region,revenue_kes,orders,share_of_revenue,revenue_rank
branch,,,,,
Nairobi,Central,26106100,376,0.363633,1
Mombasa,Coast,14003700,186,0.195058,2
Nakuru,Rift Valley,11970400,162,0.166736,3
Kisumu,Western,10868100,159,0.151382,4
Eldoret,North Rift,8844200,117,0.123191,5


,In-store,Online,Phone order
branch,,,
Eldoret,4885300,3450700,508200
Kisumu,6468600,3370800,1028700
Mombasa,8039600,4086800,1877300
Nairobi,14808400,7527400,3770300
Nakuru,6767300,4350400,852700


**Answer:** All the Excel totals match the pandas results exactly. The SUMIFS branch table, the COUNTIFS order counts, the PivotTable (branch by channel) and its grand total of KES 71,792,500 are identical to the notebook. The branch ranking in the workbook (Nairobi, Mombasa, Nakuru, Kisumu, Eldoret) is also the same as in Task 3.

**How the workbook is built:**
- `Sales`: the 1,000 orders, a `revenue` formula (`=E2*G2`) and an XLOOKUP formula that fetches each order's region from the `Branches` sheet.
- `Branches`: the branch-to-region lookup table.
- `Branch Summary`: SUMIFS revenue, COUNTIFS orders, share of revenue, rank and an XLOOKUP region for each branch.
- `PivotTable`: a PivotTable of revenue with branches as rows and channels as columns.